In [1]:
import chemsource
import pandas as pd
from ast import literal_eval
import sys
import os
sys.path.append(os.path.abspath("../extra_src"))
from harmonization import filter_synonym_list, preprocess_chemical
import pandarallel
pandarallel.pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
rosmap_data_new = pd.read_csv("../data/extra/rosmap_all_annotation_cleaned (1).csv")
rosmap_data_new["synonyms"] = rosmap_data_new["synonyms"].apply(lambda x: literal_eval(x) if pd.notna(x) else [])

In [8]:
rosmap_data_new["synonyms"] = rosmap_data_new["synonyms"].apply(lambda x: filter_synonym_list(x))
rosmap_data_new["synonyms"] = rosmap_data_new["synonyms"].apply(lambda x: preprocess_chemical(x))

In [9]:
def retrieve_text_synonyms_list(synonyms_list, model, source_priority="WIKIPEDIA"):
    """
    Retrieve text from a list of synonyms.
    
    Args:
        synonyms_list (list): A list of synonyms.
    
    Returns:
        tuple: A tuple containing the best synonym, the source of text, and the text itself.
    """
    
    results = []
    for synonym in synonyms_list:
        if isinstance(synonym, str):
            source, text = model.retrieve(synonym)
            if source == source_priority:
                return synonym, source, text
            results.append((synonym, source, text))

    results_filtered = [result for result in results if result[2] != "NO_RESULTS"]

    if results_filtered:
        return results_filtered[0]
    else:
        return None, None, None


In [4]:
openai_api_key = open("../secrets/openai_api_key.txt").read().strip()
ncbi_api_key = open("../secrets/ncbi_api_key.txt").read().strip()

In [ ]:
chem = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)


In [11]:
rosmap_data_new["text"] = rosmap_data_new["synonyms"].parallel_apply(lambda x: retrieve_text_synonyms_list(x, chem)[2] if isinstance(x, list) else None)

/opt/homebrew/anaconda3/envs/chemsource-data-analysis/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /opt/homebrew/anaconda3/envs/chemsource-data-analysis/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
/opt/homebrew/anaconda3/envs/chemsource-data-analysis/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem,

In [13]:
rosmap_data_new.to_csv("../data/extra/rosmap_all_annotation_cleaned_with_text_1.csv", index=False)

In [3]:
rosmap_with_text_1 = pd.read_csv("../data/extra/rosmap_all_annotation_cleaned_with_text_1.csv")
rosmap_with_text_1["synonyms"] = rosmap_with_text_1["synonyms"].apply(lambda x: literal_eval(x) if x !="[]" else None)
rosmap_with_text_1_none = rosmap_with_text_1[(rosmap_with_text_1["text"].isna()) & (rosmap_with_text_1["synonyms"].notna())]
rosmap_with_text_1_none["synonyms"] = rosmap_with_text_1_none["synonyms"].apply(lambda x: x[:5])

rosmap_with_text_1_available = rosmap_with_text_1[rosmap_with_text_1["text"].notna()]
# rosmap_with_text_1_none["text"] = rosmap_with_text_1_none["synonyms"].parallel_apply(lambda x: retrieve_text_synonyms_list(x, chem)[2] if isinstance(x, list) else None)

/var/folders/7f/td1j1ghj1f77gp6tfcmnv2lw0000gn/T/ipykernel_96487/2204427296.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rosmap_with_text_1_none["synonyms"] = rosmap_with_text_1_none["synonyms"].apply(lambda x: x[:5])


In [4]:
rosmap_with_text_1_available

,compound_name,input_name,scan,smiles,synonyms,text
0,L-kynurenine,Spectral Match to L-Kynurenine from NIST14,7433,Nc1ccccc1C(=O)C[C@H](N)C(=O)O,"[Kynurenine, 2-amino-4-(2-aminophenyl)-4-oxobu...",l-Kynurenine is a metabolite of the amino acid...
1,Phenylacetylglutamine,PHENYLACETYL-GLUTAMINE,7600,NC(=O)CC[C@H](NC(=O)Cc1ccccc1)C(=O)O,"[Phenylacetylglutamine, Phenylacetyl l-glutami...",Phenylacetylglutamine is a product formed by t...
3,L-histidine,L-HISTIDINE - 40.0 eV,1759,N[C@@H](Cc1cnc[nH]1)C(=O)O,"[Histidine, H-his-oh, Glyoxaline-5-alanine, An...",Histidine (symbol His or H) is an essential am...
4,L-methionine,L-methionine CollisionEnergy:102040,6738,CSCC[C@H](N)C(=O)O,"[Methionine, H-met-oh, 2-amino-4-(methylthio)b...",Methionine (symbol Met or M) () is an essentia...
8,L-tryptophan,TRYPTOPHAN,1252,N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O,"[Tryptophan, H-trp-oh, Tryptophane, Trofan, Tr...",Tryptophan (symbol Trp or W) is an α-amino aci...
...,...,...,...,...,...,...
531,UNDECANEDIOIC ACID,"1,11-undecanedicarboxylic acid (known isomers:...",9053,O=C(O)CCCCCCCCCC(=O)O,"[Hendecanedioic acid, Undecanedionic acid, Und...",Acne is an inflammatory disorder with a high ...
540,gamma-Linolenic acid,"(6Z,9Z,12Z)-octadeca-6,9,12-trienoic-acid (kno...",7111,CCCCC/C=C\C/C=C\C/C=C\CCCCC(=O)O,"[Gamma-linolenic acid, (6z,9z,12z)-octadeca-6,...",γ-Linolenic acid or GLA (INN: gamolenic acid) ...
545,gamma-Linolenic acid,"(6Z,9Z,12Z)-octadeca-6,9,12-trienoic-acid (kno...",314,CCCCC/C=C\C/C=C\C/C=C\CCCCC(=O)O,"[Gamma-linolenic acid, (6z,9z,12z)-octadeca-6,...",γ-Linolenic acid or GLA (INN: gamolenic acid) ...
548,Arachidic acid,icosanoic-acid (known isomers: 0; isobaric pea...,1175,CCCCCCCCCCCCCCCCCCCC(=O)O,"[Arachidic acid, Icosanoic acid, Arachic acid,...","Arachidic acid, also known as icosanoic acid, ..."


In [7]:
rosmap_with_text_1_available["classification"] = rosmap_with_text_1_available.parallel_apply(lambda row: chem.classify(row["compound_name"], row["text"]), axis=1)

/var/folders/7f/td1j1ghj1f77gp6tfcmnv2lw0000gn/T/ipykernel_96487/61773380.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rosmap_with_text_1_available["classification"] = rosmap_with_text_1_available.parallel_apply(lambda row: chem.classify(row["compound_name"], row["text"]), axis=1)


In [9]:
rosmap_with_text_1_available.to_parquet("../data/extra/rosmap_all_annotation_cleaned_classified.pq", index=False)